# R06 · Yocto II - caching, kernels, modules, SDKs, licenses and updates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R06_yocto_ii_caching_kernels_modules_sdks_licenses_a.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 65 min

**Skill Hub concepts:** Shared state cache and downloads directory; Kernel recipe and configuration fragments; Out-of-tree kernel module recipe; Software development kit and devtool; Reproducible builds, licenses and software bill of materials; Yocto versus Buildroot; Over-the-air update for embedded Linux

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R06")

## Map: running Yocto like a product team

| Piece | What it is | Why you care |
|---|---|---|
| `DL_DIR` | every fetched source tarball and git clone | share between builds and machines; enables offline builds |
| `SSTATE_DIR` | output of each task, keyed by a **signature** = hash of the task's inputs and its dependencies' signatures | change one recipe and only tasks whose signature changed rerun; share via a mirror for CI |
| Kernel recipe (`linux-yocto`, vendor kernels) | `.bbappend` + `SRC_URI += "file://imu.cfg"` config fragments, patches, device trees | small, reviewable kernel changes; fragments merge in order, later wins |
| Out-of-tree module | `inherit module`, a Kbuild Makefile, `RPROVIDES:${PN} += "kernel-module-<name>"` | your driver built against the exact kernel in the image |
| SDK / eSDK | `bitbake <image> -c populate_sdk` gives a cross toolchain + sysroot | app developers build without Yocto |
| `devtool` | `devtool modify/build/deploy-target <recipe>` | edit source in a workspace, push to the board over SSH |
| Reproducibility | `SOURCE_DATE_EPOCH`, fixed paths: same inputs give bit-identical images | verifiable releases, cacheable builds |
| Licenses and SBOM | `INCOMPATIBLE_LICENSE = "GPL-3.0*"`, license manifest, `create-spdx` (SPDX SBOM) | compliance and security audits |
| OTA updates | A/B root filesystems with RAUC, SWUpdate or Mender; bootloader counter rolls back a bad update | a failed update never bricks a robot |

| | Yocto | Buildroot |
|---|---|---|
| Model | layers + recipes, package feeds, many images from one config | one config, one image, simple Makefiles |
| Learning curve | steep | gentle |
| Best for | products with variants, long maintenance, vendor BSP support | small fixed appliances, fast prototypes |

Kernel config fragments look like `.config` lines: `CONFIG_IIO=y`, `CONFIG_SPI_STM32=m`,
`# CONFIG_DEBUG_INFO is not set`.

In [ ]:
import hashlib

## Task 1: task signatures and what reruns
`tasks = {name: {"inputs": [...], "deps": [...]}}` in dependency order. Signature = SHA-256 hex of the sorted inputs
joined by `"|"`, then `"|"`, then the dependencies' signatures joined by `"|"` in the listed order.

In [ ]:
def signatures(tasks):
    raise NotImplementedError  # TODO

def rerun(old, new):
    raise NotImplementedError  # TODO

lab.check("sstate", signatures, rerun)

## Task 2: merge kernel config fragments
Return `{option: value}` where value is `"y"`, `"m"`, `"n"` (from `# CONFIG_X is not set`) or the string after `=`.

In [ ]:
def merge_config(base, *fragments):
    raise NotImplementedError  # TODO

lab.check("fragments", merge_config)

## Task 3: recipe for your own kernel module

In [ ]:
%%writefile rdl-led-mod_0.1.bb
SUMMARY = "Out-of-tree LED driver for the robotics daily labs"
# TODO: LICENSE (kernel modules: GPL-2.0-only), LIC_FILES_CHKSUM, inherit module, SRC_URI with the
#       Makefile, the C file and COPYING, S, and RPROVIDES so images can install kernel-module-rdl-led

In [ ]:
lab.check("module_recipe")

## Checkpoint: Math
**M1 (Analyze).** You change only a compiler flag used by `do_compile` of one recipe. Of its tasks fetch, unpack, patch,
configure, compile, install, package: how many rerun?

**M2 (Apply).** eMMC layout for A/B updates: two 512 MB root filesystems and a 1024 MB data partition. Total MB?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** A small team ships one robot controller board, one image, rarely updated, and wants results this
week. Yocto or Buildroot?
- a) Yocto, always
- b) Buildroot is reasonable; Yocto pays off with variants, long maintenance, vendor BSPs and package feeds
- c) neither, use Ubuntu

**K2 (Understand).** Describe an A/B over-the-air update from download to rollback, and why the data partition is
separate.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r06.h`).
3. In `roboprep/checks/r06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()